# Phase A — Learn the machinery on tiny data

WIR 2026 · Notebook 1 (`pyterrier-intro`). Goal: understand indexing, the lexicon, and TF / TF-IDF on the small `ai.json` dataset before touching the 870k LongEval-Sci papers.

See roadmap §5 Phase A.

In [4]:
import pandas as pd
import pyterrier as pt

if not pt.java.started():
    pt.java.init()

## 1. Load & clean `ai.json`
Drop columns that are >50% empty; keep useful fields (`id, tags, label, description, year, author`).

In [5]:
import json

# ai.json is a BibSonomy export: an object with "types"/"properties"/"items".
# The actual records (1000 Bookmarks + 1000 Publications) live under "items".
with open("../data/ai.json", encoding="utf-8") as f:
    items = json.load(f)["items"]

df = pd.DataFrame(items)
df = df.loc[:, df.isna().mean() < 0.5]   # drop columns that are >50% empty
df.head()

,type,id,tags,intraHash,label,user,description,date,changeDate,count,url
0,Bookmark,https://www.bibsonomy.org/url/9a95a0ec1b661c27...,"[expert_system_prescription_support, ai, medic...",9a95a0ec1b661c277fba06ebd5044172,Artifical Intelligence Systems in Clinical Pra...,jepcastel,,2008-12-24 15:46:21,2012-11-25 18:07:27,1,http://www.openclinical.org/aisinpractice.html
1,Bookmark,https://www.bibsonomy.org/url/4faee58023ed540b...,"[OECD, artificial_intelligence]",4faee58023ed540b8a11e3471b6e1158,OECD Artificial Intelligence Papers,meneteqel,The OECD Artificial Intelligence Papers series...,2025-08-11 16:09:59,2025-08-11 16:09:59,1,https://www.oecd.org/en/publications/oecd-arti...
2,Bookmark,https://www.bibsonomy.org/url/ab1853d0737e8f05...,"[robotics, artificial_intelligence, European_C...",ab1853d0737e8f0565b737463d7d05ed,Artificial Intelligence | Shaping Europe’s dig...,meneteqel,The European Commission puts forward a Europea...,2020-02-20 10:38:30,2020-02-20 10:38:30,1,https://ec.europa.eu/digital-single-market/en/...
3,Bookmark,https://www.bibsonomy.org/url/11ed70d879ab786e...,"[ethics, machinelearning, learninganalytics, h...",11ed70d879ab786edaf38b4a41db3277,Artificial Intelligence in Higher Education – ...,ereidt,“The Promise and Peril of Artificial Intellige...,2018-08-26 11:31:16,2018-08-26 11:42:28,1,https://medium.com/@BryanFendley/artificial-in...
4,Bookmark,https://www.bibsonomy.org/url/0791349ca5df61f2...,"[compare, Studium, ai, vergleich, master, ki]",0791349ca5df61f2f740f401805d0b96,Best Masters of Science (MScs) in Artificial I...,hotho,Contact Schools Directly - Compare 27 Masters ...,2019-08-16 18:14:29,2019-08-16 18:14:29,1,https://www.masterstudies.com/MSc/Artificial-I...


## 2-3. Give PyTerrier the two columns it requires: `docno` and `text`

In [6]:
df["docno"] = df["id"].astype(str)
df["text"] = df["label"]

## 4. Index it

In [7]:
import os

# Terrier resolves relative index paths against its internal ./var dir, so give it an absolute one.
index_path = os.path.abspath("../ai_index")
indexer = pt.IterDictIndexer(index_path, overwrite=True,
                             meta={"docno": 200, "text": 4096})
index_ref = indexer.index(df.to_dict(orient="records"))
index = pt.IndexFactory.of(index_ref)
print(index.getCollectionStatistics().toString())

17:41:17.592 [main] ERROR org.terrier.structures.indexing.Indexer -- Could not rename index
java.io.IOException: Rename of index structure file 'c:\Users\rafae\OneDrive\Desktop\TH_Koeln\03-Semester\Web_Information_Retrieval\WIR_Retriever_Engine\ai_index/data_1.direct.bf' (exists) to 'c:\Users\rafae\OneDrive\Desktop\TH_Koeln\03-Semester\Web_Information_Retrieval\WIR_Retriever_Engine\ai_index/data.direct.bf' (exists) failed - likely that source file is still open. Possible indexing bug?
	at org.terrier.structures.IndexUtil.renameIndex(IndexUtil.java:379)
	at org.terrier.structures.indexing.Indexer.index(Indexer.java:388)
Number of documents: 2000
Number of terms: 2915
Number of postings: 12318
Number of fields: 0
Number of tokens: 12624
Field names: []
Positions:   false



## 5. Inspect the inverted index (the TDM/lexicon)
For each term read `Nt` (in how many docs) and `TF` (total frequency).

In [8]:
for kv in index.getLexicon():
    print(kv.getKey(), "->", kv.getValue().toString())

0 -> term436 Nt=8 TF=9 maxTF=2 @{0 0 0}
00 -> term669 Nt=1 TF=1 maxTF=1 @{0 13 7}
000 -> term2876 Nt=1 TF=1 maxTF=1 @{0 15 7}
0003 -> term1291 Nt=1 TF=1 maxTF=1 @{0 18 5}
01 -> term33 Nt=1 TF=1 maxTF=1 @{0 21 1}
02 -> term514 Nt=2 TF=2 maxTF=1 @{0 21 7}
03 -> term109 Nt=1 TF=1 maxTF=1 @{0 26 5}
04 -> term130 Nt=2 TF=2 maxTF=1 @{0 27 7}
05 -> term588 Nt=3 TF=3 maxTF=1 @{0 31 5}
06 -> term715 Nt=1 TF=1 maxTF=1 @{0 35 7}
07 -> term133 Nt=3 TF=3 maxTF=1 @{0 37 7}
08 -> term1006 Nt=3 TF=3 maxTF=1 @{0 42 5}
09 -> term593 Nt=6 TF=6 maxTF=1 @{0 46 5}
1 -> term143 Nt=10 TF=13 maxTF=2 @{0 54 7}
10 -> term644 Nt=6 TF=7 maxTF=2 @{0 70 6}
10th -> term2031 Nt=2 TF=2 maxTF=1 @{0 80 5}
11 -> term513 Nt=4 TF=4 maxTF=1 @{0 85 3}
12 -> term266 Nt=4 TF=4 maxTF=1 @{0 92 7}
12th -> term679 Nt=2 TF=2 maxTF=1 @{0 102 1}
13 -> term635 Nt=6 TF=6 maxTF=1 @{0 106 7}
14 -> term657 Nt=4 TF=5 maxTF=2 @{0 118 1}
14th -> term698 Nt=1 TF=1 maxTF=1 @{0 123 2}
15 -> term634 Nt=8 TF=8 maxTF=1 @{0 125 2}
15th -> term678 Nt

## 6-7. First retriever , then the built-in TF_IDF

In [26]:
tf = pt.terrier.Retriever(index, wmodel="Tf")

tf_df = tf.search("machine learning")
tf_df.head()

,qid,docid,docno,rank,score,query
0,1,1420,https://www.bibsonomy.org/bibtex/2aac90bd39bb6...,0,4.0,machine learning
1,1,1474,https://www.bibsonomy.org/bibtex/27cda56873f7e...,1,4.0,machine learning
2,1,1454,https://www.bibsonomy.org/bibtex/26433634cd767...,2,3.0,machine learning
3,1,20,https://www.bibsonomy.org/url/c78a2109d1deaefa...,3,2.0,machine learning
4,1,152,https://www.bibsonomy.org/url/73ace966c8daa12f...,4,2.0,machine learning


In [27]:
import numpy as np

N = index.getCollectionStatistics().getNumberOfDocuments()
lex = index.getLexicon()

tfidf = pt.terrier.Retriever(index, wmodel="TF_IDF")  # TF_IDF is a built-in weighting model in Terrier, so we can just specify it by name.
tfidf_df = tfidf.search("machine learning")
tfidf_df.head()

,qid,docid,docno,rank,score,query
0,1,1420,https://www.bibsonomy.org/bibtex/2aac90bd39bb6...,0,6.590638,machine learning
1,1,163,https://www.bibsonomy.org/url/80e974da88825bdb...,1,6.062074,machine learning
2,1,1474,https://www.bibsonomy.org/bibtex/27cda56873f7e...,2,5.653492,machine learning
3,1,1121,https://www.bibsonomy.org/bibtex/21f547191fa6e...,3,5.632668,machine learning
4,1,1178,https://www.bibsonomy.org/bibtex/241ecbc00c5f2...,4,5.632668,machine learning


In [28]:
tfidf_df["docno"][0]

'https://www.bibsonomy.org/bibtex/2aac90bd39bb6cbd5ccaa932633c2f98f/ijtsrd'